# Session 6 — Artificial Neural Networks
**Sebastián Ríos Saucedo · Computer Vision & Speech Recognition · EADA**

Class notebook, adapted to run in my own `Milk10k-project` repo. I commented the code and the outputs, and did the exercises at the end (mini challenge + homework before Session 7).

**What I changed so it runs on my machine**
- The class code imports `common.config` and `common.milk10k`, which aren't in my repo. My equivalents are `milk10k_pipeline.config` (paths, image folder) and the leak-free lesion-level splits I saved in Milestone 1 (`outputs/splits/{train,val,test}.csv`, seed 42).
- Everything this notebook writes goes to `outputs/session_06/`, inside the repo.
- The class compares against Session 5's random forest stored in `session_05_classical_ml/_output/baseline_scores.json`. I don't have that file, so I retrain that baseline (random forest on the same 33 features, same split) inside this notebook. That keeps the comparison fair.
- Because my split is different from the professor's (3,742 / 749 / 749 lesions rather than 3,593 / 599 / 1,048), the MILK10k numbers in sections 7–11 differ from the ones in the class comments. I replaced those comments with my own results.

#### 0. Setup

In [ ]:
import sys
import time
import json
import copy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Find the repo root by walking up from the current folder until we see milk10k_pipeline/config.py
# (the class version looked for common/config.py, which my repo doesn't have).
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / "milk10k_pipeline" / "config.py").is_file())
sys.path.insert(0, str(ROOT))

from milk10k_pipeline import config     # data only: DATA_DIR, IMAGE_DIR, image_path(), SPLIT_DIR

OUT = config.OUTPUT_DIR / "session_06"  # everything this notebook writes goes here
OUT.mkdir(parents=True, exist_ok=True)

np.set_printoptions(precision=3, suppress=True)   # print arrays with 3 decimals, no scientific notation
print(ROOT)
print(config.describe())

### 1. One neuron, by hand → in NumPy

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))   # squashes any real number into (0, 1) -> reads as a probability

def neuron(x, w, b):
    z = np.dot(x, w) + b          # 1. weighted sum: z = w1*x1 + w2*x2 + b
    return z, sigmoid(z)          # 2. activation: turn the score into a probability

x = np.array([0.6, 0.2])          # one lesion, two features: edge density, color asymmetry (normalized 0-1)
w = np.array([1.5, -0.5])         # weights: edge density pushes towards "malignant", asymmetry slightly against
b = -0.4                          # bias: shifts the threshold
neuron(x, w, b)                   # by hand: z = 1.5*0.6 - 0.5*0.2 - 0.4 = 0.4 ; sigmoid(0.4) = 0.599

In [ ]:
# Evaluate the same neuron on a 200x200 grid of (x1, x2) values to see its whole "map" of outputs.
g1, g2 = np.meshgrid(np.linspace(0, 1, 200), np.linspace(0, 1, 200))
_, p = neuron(np.stack([g1, g2], axis=-1), w, b)       # np.dot over the last axis -> works on the whole grid at once

plt.contourf(g1, g2, p, levels=20, cmap="RdBu_r", alpha=0.8)   # colour = neuron output (blue ~0, red ~1)
plt.colorbar(label="neuron output")
plt.contour(g1, g2, p, levels=[0.5], colors="k")       # the decision boundary: output 0.5 <=> z = 0 (a straight line)
plt.scatter(*x, c="k", marker="*", s=200)              # our lesion: 0.599, just on the "malignant" side
plt.xlabel("x1 = edge density"); plt.ylabel("x2 = color asymmetry");

### 2. XOR — what one line cannot do

In [ ]:
# XOR: output 1 only when exactly one input is 1. The two classes sit on opposite corners.
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([0, 1, 1, 0])

plt.figure(figsize=(3.5, 3.5))
plt.scatter(X_xor[:, 0], X_xor[:, 1], c=y_xor, cmap="coolwarm", s=400, edgecolors="k")
plt.xlabel("x1"); plt.ylabel("x2"); plt.title("XOR");

In [ ]:
def relu(z):
    return np.maximum(0, z)              # ReLU: keep positives, set negatives to 0 (the non-linearity)

# Hand-picked weights for a 2-2-1 network that solves XOR
W1 = np.array([[1, 1],
               [1, 1]], dtype=float)     # column j = weights of hidden neuron j -> both compute x1 + x2
b1 = np.array([0, -1], dtype=float)      # h1 = relu(x1 + x2),  h2 = relu(x1 + x2 - 1)  (h2 fires only for (1,1))
W2 = np.array([1, -2], dtype=float)      # output = h1 - 2*h2 -> subtracts the (1,1) case back to 0

H = relu(X_xor @ W1 + b1)                # hidden layer: 4 inputs x 2 neurons (matrix product does all 4 at once)
out = H @ W2
pd.DataFrame({"x1": X_xor[:, 0], "x2": X_xor[:, 1],
              "h1": H[:, 0], "h2": H[:, 1], "output": out, "XOR": y_xor})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
# Left: original inputs. Right: the same 4 points after the hidden layer (h1, h2).
axes[0].scatter(X_xor[:, 0], X_xor[:, 1], c=y_xor, cmap="coolwarm", s=400, edgecolors="k")
axes[0].set_title("input space: no line works"); axes[0].set_xlabel("x1"); axes[0].set_ylabel("x2")
axes[1].scatter(H[:, 0], H[:, 1], c=y_xor, cmap="coolwarm", s=400, edgecolors="k")
h1_line = np.linspace(-0.2, 2.2, 10)
axes[1].plot(h1_line, (h1_line - 0.5) / 2, "k--")       # output = h1 - 2*h2 = 0.5  ->  h2 = (h1 - 0.5) / 2
axes[1].set_title("hidden space: one line works"); axes[1].set_xlabel("h1"); axes[1].set_ylabel("h2")
plt.tight_layout()

In [ ]:
# What happens WITHOUT the ReLU? Two linear layers in a row...
linear_out = (X_xor @ W1 + b1) @ W2               # same network, NO ReLU
collapsed = X_xor @ (W1 @ W2) + (b1 @ W2)          # ...are exactly ONE linear layer with W = W1 @ W2, b = b1 @ W2
print(linear_out, collapsed)                       # identical, and not XOR
print(W1 @ W2, b1 @ W2)                            # the single equivalent layer: weights [-1, -1], bias 2

### 3. Forward pass, softmax, cross-entropy

In [ ]:
def forward(x, W1, b1, W2, b2):
    z_h = x @ W1 + b1          # hidden pre-activations (weighted sums)
    h = relu(z_h)              # hidden activations
    z_o = h @ W2 + b2          # output pre-activation
    return sigmoid(z_o), h, z_h   # probability of "malignant" + intermediate values (needed later for backprop)

# A 2-2-1 network with small arbitrary weights (as at initialisation)
W1 = np.array([[0.5, -0.3],
               [0.3,  0.4]]); b1 = np.array([0.1, -0.3])
W2 = np.array([0.4, -0.2]);   b2 = 0.1

y_hat, h, z_h = forward(np.array([0.6, 0.2]), W1, b1, W2, b2)
print(z_h, h, y_hat)

In [ ]:
def softmax(z):
    e = np.exp(z - z.max())    # subtract the max first: same result (it cancels out), but no overflow for big logits
    return e / e.sum()         # normalise -> positive numbers that sum to 1

logits = np.array([1.0, 2.0, 0.5])          # raw scores for Benign, Malignant, Indeterminate
print(np.exp(logits))                       # step 1: exponentiate (makes everything positive, exaggerates gaps)
print(softmax(logits))                      # step 2: divide by the sum -> probabilities

In [ ]:
def cross_entropy(y_true_onehot, probs, eps=1e-9):
    probs = np.clip(probs, eps, 1 - eps)                 # avoid log(0) = -inf
    return -np.sum(y_true_onehot * np.log(probs))        # with one-hot truth this is just -log(p of the true class)

y_true = np.array([0, 1, 0])                             # truth: Malignant (middle class)
for probs in ([0.10, 0.70, 0.20],                        # right, fairly confident
              [0.005, 0.99, 0.005],                      # right, very confident
              [0.80, 0.10, 0.10]):                       # wrong, confident
    print(probs, "->", round(cross_entropy(y_true, np.array(probs)), 3))

In [ ]:
# The loss curve: -log(p) as a function of the probability given to the true class
p = np.linspace(0.01, 1, 200)
plt.plot(p, -np.log(p), color="#F08200", lw=3)
for q in (0.10, 0.70, 0.99):
    plt.scatter(q, -np.log(q), s=80, color="k", zorder=3)
    plt.annotate(f"p={q} -> {-np.log(q):.2f}", (q, -np.log(q)), xytext=(8, 8), textcoords="offset points")
plt.xlabel("probability given to the TRUE class"); plt.ylabel("loss = -log(p)");

### 4. Gradient descent by hand

In [ ]:
# Toy loss with one weight: L(w) = (w - 3)^2 -> minimum at w = 3
def descend(lr, w=0.0, steps=15):
    path = [w]
    for _ in range(steps):
        grad = 2 * (w - 3)       # derivative of (w - 3)**2 = slope at the current w
        w = w - lr * grad        # one step downhill: move against the slope, scaled by the learning rate
        path.append(w)
    return np.array(path)

path = descend(lr=0.1)
print(path[:4])                  # first steps: each one covers 20% of the remaining distance to 3
print(path[-1])                  # after 15 steps -> close to 3

In [ ]:
# Same descent with three learning rates
ws = np.linspace(-4, 10, 200)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5), sharey=True)
for ax, lr, title in zip(axes, [0.01, 0.1, 1.1], ["too small", "good", "too big"]):
    path = descend(lr, steps=8 if lr > 1 else 15)
    ax.plot(ws, (ws - 3) ** 2, color="#BBBBBB")                   # the loss curve
    ax.plot(path, (path - 3) ** 2, "o-", color="#F08200")         # where gradient descent went
    ax.set_title(f"lr = {lr}: {title}  (final w = {path[-1]:.2f})")
    ax.set_ylim(-2, 60); ax.set_xlabel("w")
axes[0].set_ylabel("loss");

### 5. Backpropagation by hand — then let PyTorch check us

In [ ]:
# Back to the single neuron of section 1 (x, w, b). The lesion is malignant -> target y = 1.
y = 1.0
z, y_hat = neuron(x, w, b)
loss = -np.log(y_hat)                         # binary cross-entropy when y = 1
print(f"forward : z={z:.3f}  y_hat={y_hat:.3f}  loss={loss:.3f}")

dz = y_hat - y                                # dLoss/dz for sigmoid + cross-entropy simplifies to (prediction - truth)
grad_w = dz * x                               # chain rule: dLoss/dw = dLoss/dz * dz/dw = dz * x
grad_b = dz * 1                               # dz/db = 1
print(f"backward: dz={dz:.3f}  grad_w={grad_w}  grad_b={grad_b:.3f}")

In [ ]:
# One gradient-descent update with lr = 1, then re-run the forward pass
lr = 1.0
w_new = w - lr * grad_w                       # negative gradient -> weights go UP
b_new = b - lr * grad_b
z2, y_hat2 = neuron(x, w_new, b_new)
print(w_new, round(b_new, 3))
print(f"y_hat {y_hat:.3f} -> {y_hat2:.3f}   loss {loss:.3f} -> {-np.log(y_hat2):.3f}")

In [ ]:
import torch
import torch.nn as nn

# Same neuron in PyTorch: autograd should reproduce our hand-computed gradients
xt = torch.tensor([0.6, 0.2])
wt = torch.tensor([1.5, -0.5], requires_grad=True)   # track gradients for these
bt = torch.tensor(-0.4, requires_grad=True)

y_hat_t = torch.sigmoid(xt @ wt + bt)                # forward, same formula
loss_t = -torch.log(y_hat_t)
loss_t.backward()                                    # backpropagation, automatic -> fills wt.grad and bt.grad

print(y_hat_t.item(), loss_t.item())
print(wt.grad, bt.grad)

### 6. A real (tiny) network learns a curved boundary

In [ ]:
from sklearn.datasets import make_moons

# Two interleaving half-moons: a toy dataset that no straight line can separate
Xm, ym = make_moons(n_samples=400, noise=0.2, random_state=0)
Xm_t = torch.tensor(Xm, dtype=torch.float32)          # PyTorch layers expect float32
ym_t = torch.tensor(ym)                               # class labels as int64 (what CrossEntropyLoss wants)

def make_net(hidden):
    torch.manual_seed(0)                              # same random initial weights every time -> fair comparison
    if hidden == 0:
        return nn.Linear(2, 2)                        # no hidden layer = (multinomial) logistic regression
    return nn.Sequential(nn.Linear(2, hidden), nn.ReLU(), nn.Linear(hidden, 2))

net = make_net(8)
net

In [ ]:
def train(net, X, y, epochs=1000, lr=0.01):
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()      # softmax + cross-entropy in one (takes raw logits)
    for epoch in range(epochs):          # full-batch: all 400 points at every step
        optimizer.zero_grad()            # 1. forget old gradients (PyTorch accumulates them otherwise)
        loss = loss_fn(net(X), y)        # 2. forward + loss
        loss.backward()                  # 3. backprop
        optimizer.step()                 # 4. w <- w - lr * grad  (Adam's version)
    return loss.item()

def plot_boundary(net, ax, title):
    # Probability of class 1 over a grid -> coloured background = what the network learned
    g1, g2 = np.meshgrid(np.linspace(-2, 3, 200), np.linspace(-1.5, 2, 200))
    grid = torch.tensor(np.c_[g1.ravel(), g2.ravel()], dtype=torch.float32)
    with torch.no_grad():                # no gradients needed, just predictions
        p = torch.softmax(net(grid), dim=1)[:, 1].reshape(g1.shape)
    ax.contourf(g1, g2, p, levels=20, cmap="RdBu_r", alpha=0.7)
    ax.scatter(Xm[:, 0], Xm[:, 1], c=ym, cmap="coolwarm", s=10, edgecolors="k", linewidths=0.3)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])

In [ ]:
# Same data, same training; only the number of hidden neurons changes
fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
moons_results = {}
for ax, hidden in zip(axes, [0, 2, 4, 16]):
    net = make_net(hidden)
    final_loss = train(net, Xm_t, ym_t)
    with torch.no_grad():
        acc = (net(Xm_t).argmax(1) == ym_t).float().mean().item()    # training accuracy
    moons_results[hidden] = (round(final_loss, 3), round(acc, 3))
    plot_boundary(net, ax, f"{hidden} hidden: loss {final_loss:.2f}, acc {acc:.2f}")
plt.show()
print(moons_results)     # {hidden neurons: (final loss, accuracy)}

### 7. Back to MILK10k — Session 5's 33 features

In [ ]:
from PIL import Image
from scipy import ndimage

# ---- Session 5 ---------------------------------------------------------------
def color_stats(rgb):                                   # 6 features: mean and std of R, G, B
    return np.concatenate([rgb.mean(axis=(0, 1)), rgb.std(axis=(0, 1))])

def color_histogram(rgb, bins=8):                       # 24 features: 8-bin histogram per channel, as fractions of pixels
    n_pixels = rgb.shape[0] * rgb.shape[1]
    return np.concatenate([np.histogram(rgb[:, :, c], bins=bins, range=(0, 256))[0] / n_pixels
                           for c in range(3)])

def texture_features(rgb):                              # 3 features (Session 4 filters)
    gray = rgb.mean(axis=2)
    gx, gy = ndimage.sobel(gray, axis=1), ndimage.sobel(gray, axis=0)
    local_mean = ndimage.uniform_filter(gray, size=9)
    local_sq_mean = ndimage.uniform_filter(gray**2, size=9)
    return np.array([np.hypot(gx, gy).mean(),                    # mean edge strength (Sobel magnitude)
                     (local_sq_mean - local_mean**2).mean(),      # mean local variance (9x9 window)
                     ndimage.laplace(gray).var()])               # sharpness (variance of the Laplacian)

def extract_features(isic_id):                          # 6 + 24 + 3 = 33 numbers per image
    rgb = np.array(Image.open(config.image_path(isic_id)).convert("RGB").resize((200, 150)), dtype=float)
    return np.concatenate([color_stats(rgb), color_histogram(rgb), texture_features(rgb)])
# ------------------------------------------------------------------------------------------

# My Milestone 1 splits: lesion-level, leak-free, seed 42 (replaces milk10k.make_splits from class)
train_df, val_df, test_df = [pd.read_csv(config.SPLIT_DIR / f"{s}.csv") for s in ("train", "val", "test")]
train_df, val_df, test_df = [d[d["image_type"] == "dermoscopic"].reset_index(drop=True)
                             for d in (train_df, val_df, test_df)]        # one image per lesion
for d in (train_df, val_df, test_df):
    assert d["lesion_id"].is_unique                                       # sanity: really one image per lesion
assert not set(train_df.lesion_id) & (set(val_df.lesion_id) | set(test_df.lesion_id))   # no leakage

cache = OUT / "features_33.npz"                     # extracting features from ~5,000 images is slow -> cache once
if cache.exists():
    data = np.load(cache, allow_pickle=True)
else:
    t0 = time.perf_counter()
    data = {f"X_{n}": np.stack([extract_features(i) for i in d["isic_id"]])
            for n, d in [("train", train_df), ("val", val_df), ("test", test_df)]}
    np.savez(cache, **data)
    print(f"{time.perf_counter() - t0:.0f} s")

X_train, X_val, X_test = data["X_train"], data["X_val"], data["X_test"]
print(X_train.shape, X_val.shape, X_test.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler

CLASSES = ["Benign", "Indeterminate", "Malignant"]            # same order as my label_map.json (y_primary)
to_int = {c: i for i, c in enumerate(CLASSES)}
y_train = torch.tensor(train_df["diagnosis_1"].map(to_int).to_numpy())
y_val = torch.tensor(val_df["diagnosis_1"].map(to_int).to_numpy())
y_test = torch.tensor(test_df["diagnosis_1"].map(to_int).to_numpy())

# Neural nets need features on similar scales -> standardise (mean 0, std 1). Fit on TRAIN only (no leakage).
scaler = StandardScaler().fit(X_train)
Xtr = torch.tensor(scaler.transform(X_train), dtype=torch.float32)
Xva = torch.tensor(scaler.transform(X_val), dtype=torch.float32)
Xte = torch.tensor(scaler.transform(X_test), dtype=torch.float32)

print(Xtr.dtype, Xtr.shape, y_train[:10])
print(torch.bincount(y_train))                                # class counts: Benign, Indeterminate, Malignant

In [ ]:
# "balanced" class weights: n_samples / (n_classes * n_in_class) -> rare classes count more in the loss
class_weights = len(y_train) / (3 * torch.bincount(y_train).float())
class_weights

#### 7b. Session 5 baseline, retrained on my split

The class compares against Session 5's random forest (`baseline_scores.json`). I don't have that file, so I retrain the same kind of model here: a random forest on the same 33 features, the same split and the same labels. It uses `class_weight="balanced"` (same idea as the class weights above), 500 trees and `min_samples_leaf=5`. Trees don't need scaling, so it uses the raw features. I evaluate it on the test set only once and save it as `outputs/session_05/baseline_scores.json` so the scoreboard in section 10 reads it just as the class code does.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score

baseline_path = config.OUTPUT_DIR / "session_05" / "baseline_scores.json"
rf = RandomForestClassifier(n_estimators=500, class_weight="balanced", min_samples_leaf=5,
                            random_state=0, n_jobs=-1).fit(X_train, y_train.numpy())
baseline = {"model": "Random forest (500 trees, balanced, min_samples_leaf=5) on the 33 features",
            "val_macro_f1": round(float(f1_score(y_val, rf.predict(X_val), average="macro")), 3),
            "test_macro_f1": round(float(f1_score(y_test, rf.predict(X_test), average="macro")), 3)}
baseline_path.parent.mkdir(parents=True, exist_ok=True)
baseline_path.write_text(json.dumps(baseline, indent=2))
RF_VAL_F1 = baseline["val_macro_f1"]          # the number to beat (validation)
baseline

### 8. An MLP on MILK10k — mini-batches, epochs, curves

In [ ]:
def make_mlp(hidden=(64,), dropout=0.0, seed=0, d_in=33):
    # d_in added by me (default 33 = class behaviour) so I can reuse it for raw pixels in the homework
    torch.manual_seed(seed)                     # same initial weights for everyone
    layers = []
    for h in hidden:                            # one Linear + ReLU (+ optional Dropout) block per hidden layer
        layers += [nn.Linear(d_in, h), nn.ReLU()]
        if dropout:
            layers.append(nn.Dropout(dropout))
        d_in = h
    layers.append(nn.Linear(d_in, 3))           # 3 logits, no softmax (it's inside the loss)
    return nn.Sequential(*layers)

model = make_mlp()
print(model)
print(sum(p.numel() for p in model.parameters()))        # (33*64 + 64) + (64*3 + 3) = 2,371 weights

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# Mini-batches: instead of using all training images for every update, take 64 at a time (shuffled each epoch)
train_loader = DataLoader(TensorDataset(Xtr, y_train), batch_size=64, shuffle=True)
print(len(train_loader))                                  # batches per epoch = ceil(n_train / 64)
xb, yb = next(iter(train_loader))
print(xb.shape, yb.shape)                                 # one batch: 64 rows x 33 features, 64 labels

In [ ]:
def fit(model, epochs=50, lr=1e-3, weight_decay=0.0, batch_size=64, data=None):
    # data=(X_train, y_train, X_val, y_val) added by me for the homework; None = the 33-feature tensors (class behaviour)
    Xtr_, ytr_, Xva_, yva_ = data if data is not None else (Xtr, y_train, Xva, y_val)
    loader = DataLoader(TensorDataset(Xtr_, ytr_), batch_size=batch_size, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)  # weight_decay = L2 penalty
    loss_fn = nn.CrossEntropyLoss(weight=class_weights)          # weighted -> Indeterminate errors cost ~13x more
    history = []
    for epoch in range(epochs):
        model.train()                                            # training mode (dropout ON)
        total = 0.0
        for xb, yb in loader:                                    # the 4-step loop, once per mini-batch
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
            total += loss.item() * len(xb)                       # sum of per-sample losses for the epoch average
        model.eval()                                             # evaluation mode (dropout OFF)
        with torch.no_grad():                                    # after every epoch, score the validation set
            val_logits = model(Xva_)
            history.append({"epoch": epoch,
                            "train_loss": total / len(Xtr_),
                            "val_loss": loss_fn(val_logits, yva_).item(),
                            "val_f1": f1_score(yva_, val_logits.argmax(1), average="macro")})
    return pd.DataFrame(history).set_index("epoch")

t0 = time.perf_counter()
model = make_mlp()
hist = fit(model)
print(f"{time.perf_counter() - t0:.0f} s")
hist.iloc[[0, 9, 19, 29, 39, 49]].round(3)        # a few snapshots of the 50 epochs

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
hist[["train_loss", "val_loss"]].plot(ax=axes[0], color=["#F08200", "#2E6FBA"], title="loss")
hist["val_f1"].plot(ax=axes[1], color="#2E6FBA", title="validation macro-F1")
axes[1].axhline(RF_VAL_F1, color="k", ls="--", label=f"Session 5 forest ({RF_VAL_F1})")   # my forest, not the class 0.534
axes[1].legend();

In [ ]:
model.eval()
with torch.no_grad():
    val_pred = model(Xva).argmax(1)                # predicted class = highest logit
mlp_val_f1 = f1_score(y_val, val_pred, average="macro")
print(f"MLP val macro-F1 = {mlp_val_f1:.3f}   (Session 5 forest on my split: {RF_VAL_F1})")

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_predictions(y_val, val_pred, display_labels=CLASSES, cmap="Oranges");

### 9. Overfitting on purpose — and three cures

In [ ]:
# A much bigger network (two hidden layers of 256) on only ~3,700 images: it will memorise
big = make_mlp(hidden=(256, 256))
print(sum(p.numel() for p in big.parameters()))           # 33*256+256 + 256*256+256 + 256*3+3
hist_big = fit(big, epochs=100)
hist_big[["train_loss", "val_loss"]].plot(color=["#F08200", "#2E6FBA"], logy=True,
                                          title="256x256, no regularization");

In [ ]:
# Cures 1 + 2: dropout (randomly switch off 30% of hidden neurons while training) + weight decay (L2 on weights)
reg = make_mlp(hidden=(256, 256), dropout=0.3)
hist_reg = fit(reg, epochs=100, weight_decay=1e-4)

fig, ax = plt.subplots(figsize=(7, 3.5))
hist_big["val_loss"].plot(ax=ax, label="val loss, no regularization", color="#2E6FBA", logy=True)
hist_reg["val_loss"].plot(ax=ax, label="val loss, dropout + weight decay", color="#F08200")
# Cure 3: early stopping = keep the model from the epoch with the lowest validation loss
best_epoch = hist_reg["val_loss"].idxmin()
ax.axvline(best_epoch, color="k", ls=":", label=f"early stopping: epoch {best_epoch}")
ax.legend();
plt.show()
print(hist_big.iloc[-1].round(3).to_dict())     # last epoch, no regularisation
print(hist_reg.iloc[-1].round(3).to_dict())     # last epoch, dropout + weight decay
print("best val F1 during training:", round(hist_big.val_f1.max(), 3), "(big) /", round(hist_reg.val_f1.max(), 3), "(reg)")

### 10. The test set — once — and the scoreboard

In [ ]:
# The model chosen BEFORE looking at test: the class MLP 33-64-3 from section 8
with torch.no_grad():
    test_pred = model(Xte).argmax(1)
test_acc = accuracy_score(y_test, test_pred)
test_f1 = f1_score(y_test, test_pred, average="macro")
print(f"TEST accuracy = {test_acc:.3f}   TEST macro-F1 = {test_f1:.3f}")

baseline = json.loads(baseline_path.read_text())     # written in 7b (class: session_05_classical_ml/_output/...)
scoreboard = pd.DataFrame({
    "Session 5 - random forest (33 features)": [baseline["val_macro_f1"], baseline["test_macro_f1"]],
    "Session 6 - MLP 33-64-3 (33 features)": [round(mlp_val_f1, 3), round(test_f1, 3)],
}, index=["val macro-F1", "test macro-F1"]).T
scoreboard

In [ ]:
scores = {"model": "MLP 33-64-3 on Session 5's 33 handcrafted features",
          "val_macro_f1": round(float(mlp_val_f1), 3),
          "test_accuracy": round(float(test_acc), 3),
          "test_macro_f1": round(float(test_f1), 3)}
(OUT / "mlp_scores.json").write_text(json.dumps(scores, indent=2))    # saved for Session 7's scoreboard
scores

### 11. Mini challenge — tune it

Change **one thing at a time**, train, and note the final **validation** macro-F1:

1. learning rate: `lr=0.1`, then `lr=1e-5`
2. width: `hidden=(4,)`, then `hidden=(256,)`
3. depth: `hidden=(64, 64)`

Number to beat: Session 5's random forest (on my split: see 7b). The test set stays closed.

In [ ]:
# Every run starts from the class default (hidden=(64,), lr=1e-3, 50 epochs, seed 0) and changes ONE thing
experiments = {
    "default  (64,), lr=1e-3": dict(),
    "lr = 0.1":                dict(lr=0.1),
    "lr = 1e-5":               dict(lr=1e-5),
    "width (4,)":              dict(hidden=(4,)),
    "width (256,)":            dict(hidden=(256,)),
    "depth (64, 64)":          dict(hidden=(64, 64)),
}
rows, histories = [], {}
for name, cfg in experiments.items():
    cfg = dict(cfg)
    m = make_mlp(hidden=cfg.pop("hidden", (64,)))
    h = fit(m, **cfg)
    histories[name] = h
    rows.append({"experiment": name,
                 "params": sum(p.numel() for p in m.parameters()),
                 "final train loss": h.train_loss.iloc[-1],
                 "final val loss": h.val_loss.iloc[-1],
                 "final val F1": h.val_f1.iloc[-1],
                 "best val F1 (epoch)": f"{h.val_f1.max():.3f} ({h.val_f1.idxmax()})"})
challenge = pd.DataFrame(rows).set_index("experiment").round(3)
challenge["beats forest?"] = challenge["final val F1"] > RF_VAL_F1
challenge

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
for name, h in histories.items():
    h["val_loss"].plot(ax=axes[0], label=name)
    h["val_f1"].plot(ax=axes[1], label=name)
axes[0].set_title("validation loss"); axes[0].set_yscale("log")
axes[1].set_title("validation macro-F1"); axes[1].axhline(RF_VAL_F1, color="k", ls="--", label="forest")
axes[1].legend(fontsize=8, loc="lower right");

### Homework · Before Session 7

1. Add your MLP's validation and test macro-F1 to the scoreboard, next to Session 5's forest.
2. Read the docs of nn.Conv2d: which arguments are Session 4's kernel size, stride and padding?
3. Train the MLP on raw pixels (32×32×3 = 3,072 numbers). Better or worse than 33 features? Why?

#### Homework 3 — the MLP on raw pixels (done first so its result can go on the scoreboard)

In [ ]:
# Each dermoscopic image -> resized to 32x32 RGB -> flattened to 3,072 numbers. No hand-crafted features.
def raw_pixels(isic_id, size=32):
    return np.array(Image.open(config.image_path(isic_id)).convert("RGB").resize((size, size)), dtype=np.uint8)

pix_cache = OUT / "pixels_32.npz"
if pix_cache.exists():
    pix = np.load(pix_cache)
else:
    t0 = time.perf_counter()
    pix = {f"P_{n}": np.stack([raw_pixels(i) for i in d["isic_id"]])
           for n, d in [("train", train_df), ("val", val_df), ("test", test_df)]}
    np.savez_compressed(pix_cache, **pix)
    print(f"{time.perf_counter() - t0:.0f} s")

P_train, P_val, P_test = [pix[f"P_{n}"].reshape(len(pix[f"P_{n}"]), -1).astype(np.float32)
                          for n in ("train", "val", "test")]          # (n, 32, 32, 3) -> (n, 3072)
pix_scaler = StandardScaler().fit(P_train)                            # standardise every pixel position, train only
Ptr, Pva, Pte = [torch.tensor(pix_scaler.transform(a), dtype=torch.float32) for a in (P_train, P_val, P_test)]
print(Ptr.shape, Pva.shape, Pte.shape)

# Look at what the network actually gets: 8 training images at 32x32
fig, axes = plt.subplots(1, 8, figsize=(14, 2.2))
for ax, i in zip(axes, range(8)):
    ax.imshow(pix["P_train"][i]); ax.set_title(CLASSES[y_train[i]], fontsize=8); ax.axis("off")

In [ ]:
pix_data = (Ptr, y_train, Pva, y_val)

# (a) Exactly the class recipe, only the input changes: 3072-64-3
pix_mlp = make_mlp(hidden=(64,), d_in=3072)
print("params:", sum(p.numel() for p in pix_mlp.parameters()))       # 3072*64+64 + 64*3+3
t0 = time.perf_counter()
hist_pix = fit(pix_mlp, data=pix_data)
print(f"{time.perf_counter() - t0:.0f} s")

# (b) Same, with the section 9 cures (dropout + weight decay) - the fairer version for so many inputs
pix_mlp_reg = make_mlp(hidden=(64,), dropout=0.3, d_in=3072)
hist_pix_reg = fit(pix_mlp_reg, weight_decay=1e-3, data=pix_data)

pix_vs = pd.DataFrame({
    "33 features, 33-64-3":            hist.iloc[-1],
    "raw pixels, 3072-64-3":           hist_pix.iloc[-1],
    "raw pixels, 3072-64-3 + dropout/wd": hist_pix_reg.iloc[-1],
}).T.round(3)
pix_vs["params"] = [2371, sum(p.numel() for p in pix_mlp.parameters()), sum(p.numel() for p in pix_mlp_reg.parameters())]
pix_vs["best val F1 during training"] = [round(hist.val_f1.max(), 3), round(hist_pix.val_f1.max(), 3), round(hist_pix_reg.val_f1.max(), 3)]
pix_vs

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
for (name, h), c in zip([("33 features", hist), ("raw pixels", hist_pix), ("raw pixels + dropout/wd", hist_pix_reg)],
                        ["#2E6FBA", "#F08200", "#7A7A7A"]):
    h["train_loss"].plot(ax=axes[0], color=c, ls="--", label=f"{name} - train")
    h["val_loss"].plot(ax=axes[0], color=c, label=f"{name} - val")
    h["val_f1"].plot(ax=axes[1], color=c, label=name)
axes[0].set_yscale("log"); axes[0].set_title("loss"); axes[0].legend(fontsize=7)
axes[1].axhline(RF_VAL_F1, color="k", ls="--", label="forest"); axes[1].set_title("validation macro-F1"); axes[1].legend(fontsize=8);

#### Homework 1 — the scoreboard

In [ ]:
# Test set: each model below was fixed BEFORE its test score was computed; nothing was re-tuned after.
pix_mlp.eval()
with torch.no_grad():
    pix_test_f1 = f1_score(y_test, pix_mlp(Pte).argmax(1), average="macro")

scoreboard = pd.DataFrame({
    "Session 5 - random forest (33 features)":   [baseline["val_macro_f1"], baseline["test_macro_f1"]],
    "Session 6 - MLP 33-64-3 (33 features)":     [round(mlp_val_f1, 3), round(test_f1, 3)],
    "Session 6 HW - MLP 3072-64-3 (raw pixels)": [round(hist_pix.val_f1.iloc[-1], 3), round(pix_test_f1, 3)],
}, index=["val macro-F1", "test macro-F1"]).T
scoreboard.to_csv(OUT / "scoreboard.csv")
scores["raw_pixels_val_macro_f1"] = round(float(hist_pix.val_f1.iloc[-1]), 3)
scores["raw_pixels_test_macro_f1"] = round(float(pix_test_f1), 3)
(OUT / "mlp_scores.json").write_text(json.dumps(scores, indent=2))
scoreboard

#### Homework 2 — `nn.Conv2d` vs Session 4

Signature from the PyTorch docs:

```python
torch.nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0,
                dilation=1, groups=1, bias=True, padding_mode='zeros')
```

| Session 4 concept | `nn.Conv2d` argument | Notes |
|---|---|---|
| kernel size *k* (3×3 Sobel, 5×5 Gaussian…) | `kernel_size` | int (`3`) or tuple (`(3, 5)`) |
| stride *s* (how far the window jumps) | `stride` | default 1 = every position, like our `conv2d` loop |
| padding *p* (pixels added around the border) | `padding` | int, tuple, or `'same'` / `'valid'` |
| padding **type** (zero vs replicate/mirror) | `padding_mode` | `'zeros'` (default), `'reflect'`, `'replicate'`, `'circular'` |

**Same output-size formula as Session 4.** `out = floor((N + 2p − k) / s) + 1`. With p = 0 and s = 1 this reduces to Session 4's **N − k + 1**.

**Two differences from Session 4.**
1. **The kernel values are learned.** In Session 4 we picked the numbers (Sobel, Gaussian…). In `nn.Conv2d` they live in `conv.weight` (shape `out_channels × in_channels × k × k`) and backprop learns them, just like the MLP weights above. Every layer also learns **many** kernels at once (`out_channels`), and each one looks at all input channels (`in_channels`, e.g. 3 for RGB), not a single grayscale image.
2. **No kernel flip.** PyTorch's "convolution" is a cross-correlation: the kernel is not flipped. That's the same operation as `scipy.ndimage.correlate`, which I used to check my hand-written `conv2d` in Session 4.

The other arguments: `dilation` spreads the kernel taps apart (a 3×3 kernel with dilation 2 covers a 5×5 area), `groups` splits the channels into independent groups, and `bias` adds one learned constant per output channel.

The cell below checks this on a real lesion. I build an `nn.Conv2d` with `kernel_size=3, stride=1, padding=1, padding_mode='replicate'`, load the Session 4 Sobel-x kernel into its weights, and compare the result with `scipy.ndimage.correlate`. I also try stride 2 and no padding to confirm the size formula.

In [ ]:
gray = np.array(Image.open(config.image_path(train_df.isic_id[0])).convert("L").resize((200, 150)), dtype=np.float32)
sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32)    # Session 4 kernel

conv = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3, stride=1, padding=1,
                 padding_mode="replicate", bias=False)
with torch.no_grad():
    conv.weight[:] = torch.from_numpy(sobel_x).view(1, 1, 3, 3)    # hand-set instead of learned
    torch_out = conv(torch.from_numpy(gray)[None, None])[0, 0].numpy()   # input shape: (batch, channels, H, W)

scipy_out = ndimage.correlate(gray, sobel_x, mode="nearest")      # 'nearest' = replicate padding
print("conv weight shape:", tuple(conv.weight.shape), "| output:", torch_out.shape,
      "| max |torch - scipy| =", float(np.abs(torch_out - scipy_out).max()))

# Output size = floor((N + 2p - k) / s) + 1
for k, s, p in [(3, 1, 0), (3, 1, 1), (3, 2, 1), (5, 2, 0)]:
    c = nn.Conv2d(1, 1, kernel_size=k, stride=s, padding=p)
    H, W = c(torch.zeros(1, 1, 150, 200)).shape[-2:]
    print(f"k={k} s={s} p={p}: torch {H}x{W}  formula {(150 + 2*p - k)//s + 1}x{(200 + 2*p - k)//s + 1}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].imshow(gray, cmap="gray"); axes[0].set_title("input"); axes[0].axis("off")
axes[1].imshow(torch_out, cmap="RdBu_r"); axes[1].set_title("nn.Conv2d with Sobel-x weights"); axes[1].axis("off");